# Phase 2 — reference data, review and four-view experiments

**Research question:** can BPDR learn a correction that stays stable under false note-boundary changes while responding to known local detuning?

This notebook builds the experiment foundation. A/B share the original waveform; C/D share a locally detuned waveform. A/C use nominal timing boundaries; B/D use the same corrupted boundaries. No learned model is present.

Human reviews remain pending until two distinct musically experienced reviewers record their decisions. Exploratory artifacts cannot enter training. Run All uses training material for the renderer pilot; development, PP1 evaluation and final holdout are not used to select it.

In [ ]:
import os, json, sys
from pathlib import Path
from collections import Counter
import numpy as np
import soundfile as sf
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
import bpdr_input
COMPONENT = Path(bpdr_input.__file__).resolve().parents[2]
DATASET = Path(os.environ.get('BPDR_DATASET_ROOT', str(COMPONENT.parents[2]/'Input Validation'/'vocadito')))
assert DATASET.is_dir(), 'Set BPDR_DATASET_ROOT to your vocadito folder before running.'
print('Python:', sys.version.split()[0], '\nComponent:', COMPONENT, '\nDataset:', DATASET)
plt.rcParams.update({'figure.figsize': (12, 4)})

## 1. Load or build a versioned source/crop manifest

In [ ]:
from bpdr_input.dataset import build_dataset, verify_manifest, dump, reviewed_intervals, reference_mask, DataPolicy
from dataclasses import asdict
import uuid
phase2 = COMPONENT/'runs'/'phase2'
latest = phase2/'latest.json'
if latest.exists():
    DATA_RUN = Path(json.loads(latest.read_text())['path'])
    manifest = json.loads((DATA_RUN/'manifest.json').read_text())
    assert Path(manifest['dataset_root']) == DATASET.resolve(), 'Existing run uses a different dataset. Build a new data run.'
else:
    DATA_RUN = phase2/uuid.uuid4().hex
    manifest = build_dataset(DATASET, DATA_RUN)
    dump(latest, {'path':str(DATA_RUN)})
assert verify_manifest(manifest)
if manifest['policy'] != asdict(DataPolicy()):
    display(Markdown('**Historical data policy:** this immutable run uses its saved crop/split settings, which differ from the current defaults. Preserve its reviews; build a new data run for current settings.'))
else:
    print('Data-run policy matches current defaults.')
summary = json.loads((DATA_RUN/'data_summary.json').read_text())
display(summary)
print('Data run:', DATA_RUN)
print('Source-level and generated-reference checksums verified.')


## 2. Independent singer partitions — split before cropping

In [ ]:
splits = manifest['singer_splits']
for p in manifest['phrases']: assert p['split'] == splits[p['singer_id']]
counts = Counter(splits.values())
fig, ax = plt.subplots(); ax.bar(list(counts), list(counts.values())); ax.set_ylabel('Independent singers'); ax.set_title('Seed 339: fixed singer partitions'); plt.show()
display([{'source':s['source_id'],'errors':s['errors'],'warnings':s['warnings']} for s in manifest['sources'] if s['errors'] or s['warnings']])
print('Phrase counts:', dict(Counter(p['split'] for p in manifest['phrases'])))

Out-of-bounds annotations are recorded and excluded, not silently repaired. Overlaps, annotator disagreements, unvoiced reference samples and partial crop-edge notes are masked. Crops are non-overlapping source-rate sample slices, <=20 seconds, with original offsets retained. A common rest is preferred for a cut; forced partial-note edges remain flagged.

## 3. Human review — actual decisions only

In [ ]:
import csv
with (DATA_RUN/'reviews.csv').open(encoding='utf-8-sig', newline='') as f: review_rows = list(csv.DictReader(f))
review_states = {p['phrase_id']:reviewed_intervals(p,review_rows)[1] for p in manifest['phrases']}
print('Review states:', dict(Counter(review_states.values())))
print('Review sheet:', DATA_RUN/'reviews.csv')
display(review_rows[:2])

Each phrase has two rows in `reviews.csv`. Each reviewer independently listens and checks F0/A1/A2 references, timing, expressiveness and whether the baseline is acceptable in the selected interval. Record a distinct reviewer ID, musical experience, decision (`approve`, `reject`, `uncertain`), crop-local approved intervals as JSON such as `[[1.0, 2.0]]`, reason and UTC timestamp. Do not approve from automatic scores alone.

Two approvals and their interval intersection are required. Missing, stale, disagreeing or incomplete reviews block training eligibility. Partial notes and uncertain transitions stay protected even inside approved intervals. Existing A1/A2 dataset annotators do not substitute for these reviews.

## 4. Run or inspect the clearly labelled exploratory pilot

In [ ]:
from bpdr_input.laboratory import (run_interventions, validate_pair_record,
                                   require_training_example, source_code_sha256)
from bpdr_input.interventions import AuditPolicy
from bpdr_input.pipeline import checksum
display(AuditPolicy().__dict__)
pilot_pointer = DATA_RUN/'latest_exploratory.json'
saved_pilot_is_current = False
if pilot_pointer.exists():
    try:
        EXPERIMENT = Path(json.loads(pilot_pointer.read_text())['path'])
        experiment = json.loads((EXPERIMENT/'summary.json').read_text())
        locked_policy = json.loads((EXPERIMENT/'locked_audit_policy.json').read_text())
        expected_policy = json.loads(json.dumps(AuditPolicy().__dict__))
        saved_pilot_is_current = (locked_policy == expected_policy
                                  and experiment.get('source_code_sha256') == source_code_sha256()
                                  and experiment.get('data_manifest_sha256') == checksum(DATA_RUN/'manifest.json')
                                  and experiment.get('review_sheet_sha256') == checksum(DATA_RUN/'reviews.csv'))
    except (OSError, ValueError, KeyError, TypeError):
        saved_pilot_is_current = False
if not saved_pilot_is_current:
    if pilot_pointer.exists():
        display(Markdown('**Historical pilot detected:** code, data, review sheet or audit policy differs. The old run remains saved; creating a new exploratory run under current provenance.'))
    EXPERIMENT, experiment = run_interventions(DATA_RUN, exploratory=True, limit=1)
print('Exploratory run:', EXPERIMENT)
display({k:v for k,v in experiment.items() if k not in ('examples','review_states')})
assert all(r['split']=='train' and r['exploratory'] and not r['training_eligible'] for r in experiment['examples'])
for record in experiment['examples']:
    assert validate_pair_record(record)
    try: require_training_example(record)
    except ValueError: pass
    else: raise AssertionError('An exploratory example was allowed into training')


Audit policy v0.3 keeps the v0.2 fine/coarse comparison: independent YIN checks plateau shift at <=5-cent median and <=10-cent 90th-percentile error; PESTO is a coarse consistency check at <=20/40 cents. It also requires at least 10 jointly auditable frames and 75% coverage of eligible plateau frames. Outside support, it requires at least 10 auditable frames, 75% coverage, <=5-cent median drift and <=20-cent 90th-percentile drift. Failed renders stay excluded. Earlier v0.1/v0.2 results remain historical evidence, not current-policy results.

Raised-cosine waveform crossfades do not imply a known linear pitch curve. Ramp/guard frames have no supervised detuning target. Only audited plateau frames receive negative injected-cents targets; protected reliable outside frames receive zero. These private targets and references never enter model-input arrays.


## 5. Inspect one reference phrase and its provenance

In [ ]:
REVIEW_PHRASE_ID = None # Set to a phrase_id from reviews.csv to inspect another candidate.
if REVIEW_PHRASE_ID is not None:
    phrase = next(p for p in manifest['phrases'] if p['phrase_id']==REVIEW_PHRASE_ID)
    example = experiment['examples'][0] if experiment['examples'] else None
elif experiment['examples']:
    example = experiment['examples'][0]
    phrase = next(p for p in manifest['phrases'] if p['phrase_id']==example['phrase_id'])
else:
    example = None
    phrase = next(p for p in manifest['phrases'] if p['split']=='train')
display(phrase)
display(Audio(filename=phrase['audio_path']))
f0 = np.loadtxt(phrase['f0_path'], delimiter=',', ndmin=2)
notes = json.loads(Path(phrase['notes_path']).read_text())
fig, ax = plt.subplots()
ax.plot(f0[:,0], np.where(f0[:,1]>0,f0[:,1],np.nan), label='Performed F0 reference', alpha=.6)
for name,color in [('A1','orange'),('A2','green')]:
    for i,n in enumerate(notes[name]):
        ax.hlines(n['pitch_hz'],n['start_seconds'],n['start_seconds']+n['duration_seconds'],colors=color,label=name if i==0 else None)
ax.set_xlabel('Crop-local seconds'); ax.set_ylabel('Reference Hz (not proven intended pitch)'); ax.legend(); plt.show()

## 6. A/B/C/D identity, boundary change and detuned audio

The laboratory's **nominal timing track comes from A1 note onsets and offsets**; false split, merge and jitter variants change only that annotation-derived track. The Phase 1 provisional boundary score comes from measured acoustics. The plot below compares both on the same pilot phrase, but does not establish boundary accuracy or runtime robustness: the annotation track is not available to the live validator.


In [ ]:
audited_examples = [r for r in experiment['examples'] if r['audit_passed']]
if audited_examples:
    chosen = next((r for r in audited_examples if r['cents']==20 and r['corruption']=='split'), audited_examples[0])
    assert chosen['waveform_sha256']['A']==chosen['waveform_sha256']['B']
    assert chosen['waveform_sha256']['C']==chosen['waveform_sha256']['D']
    display(Markdown('**A/B: identical original waveform**')); display(Audio(filename=chosen['waveform_paths']['A']))
    display(Markdown(f"**C/D: identical waveform with {chosen['cents']:+g} cents local detuning — exploratory intervention, not repaired audio**")); display(Audio(filename=chosen['waveform_paths']['C']))
    arrays={}
    for name in 'ABCD':
        with np.load(Path(chosen['input_directory'])/(name+'_input.npz')) as z: arrays[name]={k:z[k] for k in z.files}
    t=arrays['A']['times_seconds']
    with np.load(Path(chosen['waveform_paths']['A']).parent/'original_acoustics.npz') as source_features:
        acoustic_boundary = source_features['provisional_boundary_score']
    assert len(acoustic_boundary)==len(t)
    fig,ax=plt.subplots(3,1,sharex=True,figsize=(12,8))
    ax[0].plot(t,arrays['A']['boundary_features_71'][:,-1],label='A/C nominal A1 onset+offset timing')
    ax[0].plot(t,arrays['B']['boundary_features_71'][:,-1],label=f"B/D {chosen['corruption']} timing",alpha=.7); ax[0].set_ylabel('Lab timing'); ax[0].legend()
    ax[1].plot(t,arrays['A']['boundary_features_71'][:,-1],label='A1 annotation timing',alpha=.7)
    ax[1].plot(t,acoustic_boundary,label='Phase 1 acoustic provisional score',alpha=.7); ax[1].set_ylabel('Boundary score'); ax[1].legend()
    ax[2].plot(t,np.where(arrays['A']['frame_features_70'][:,66]>.5,arrays['A']['frame_features_70'][:,64],np.nan),label='Original measured cents')
    ax[2].plot(t,np.where(arrays['C']['frame_features_70'][:,66]>.5,arrays['C']['frame_features_70'][:,64],np.nan),label='Detuned measured cents',alpha=.7); ax[2].set_ylabel('Cents relative to 440 Hz'); ax[2].set_xlabel('Crop seconds'); ax[2].legend()
    fig.tight_layout(); plt.show()
else:
    print('No audit-passing rendered pair is available; inspect saved exclusions rather than inventing an example.')


## 7. Measured realized shifts — independent denominator

In [ ]:
rows=[]
seen=set()
for r in experiment['examples']:
    if r['audit_path'] in seen: continue
    seen.add(r['audit_path'])
    audit=json.loads(Path(r['audit_path']).read_text())
    rows.append({'phrase':r['phrase_id'],'injected_cents':r['cents'],'passed':audit['passed'], 'YIN':audit['independent_yin'], 'PESTO':audit['pesto']})
display(rows)
if rows:
    fig,ax=plt.subplots()
    shifts=[r['injected_cents'] for r in rows]
    ax.plot(shifts,[r['YIN']['median_shift_cents'] for r in rows],'o',label='Independent YIN')
    ax.plot(shifts,[r['PESTO']['median_shift_cents'] for r in rows],'x',label='Coarse PESTO')
    ax.plot([-30,30],[-30,30],'--',label='Requested plateau shift'); ax.set_xlabel('Injected cents'); ax.set_ylabel('Measured median shift (cents)'); ax.legend(); plt.show()
print('Independent phrases:',experiment['independent_phrase_count'],'Independent singers:',experiment['independent_singer_count'])
print('Training eligible:',experiment['training_eligible_pairs'])

## 8. Exercise the actual review gate

In [ ]:
REVIEWED_RUN, reviewed_report = run_interventions(DATA_RUN, exploratory=False, limit=1)
display({k:v for k,v in reviewed_report.items() if k not in ('examples','review_states')})
if not any(state=='approved' for state in review_states.values()):
    assert reviewed_report['training_eligible_pairs']==0
    print('Pending reviews correctly produced no training-ready examples.')

## Phase 2 gate and next step
The laboratory executes and the notebooks expose the proposed experimental mechanism. That is not evidence of BPDR superiority. Human review remains open, reference bounds exclusions remain visible, and the current exploratory pilot is too small for an accuracy claim.

After both reviewers finish `reviews.csv`, run the review-gated interventions command without `--exploratory`. Only review-approved, audit-passing pairs can feed Phase 3. Keep the fixed singer partitions and policy; do not tune on PP1 evaluation or final holdout. Phase 3 adds matched baseline models and supervised anchors; Phase 4 adds BPDR's paired-response losses.